# Lesson 07 - Планирование шаблона проектирования

Этот блокнот демонстрирует **шаблон проектирования планирования** для ИИ-агентов с использованием Microsoft Agent Framework.
Вы узнаете, как разбивать сложные запросы на поездки на структурированные подзадачи, назначать их специализированным агентам
и выполнять полученный план — все это с помощью структурированного вывода на основе моделей Pydantic.


## Настройка


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Разбиение задачи

Разбиение задачи является основой шаблона проектирования планирования. Вместо того чтобы просить одного агента выполнить сложный запрос от начала до конца, мы делим проблему на более мелкие, четко определённые **подзадачи**. Каждая подзадача назначается специализированному агенту (например, по авиарейсам, отелям, мероприятиям) с чёткими приоритетами и порядком зависимостей.

Этот подход даёт несколько преимуществ:
- **Ясность**: у каждой подзадачи есть одна зона ответственности.
- **Параллелизм**: независимые подзадачи могут выполняться одновременно.
- **Надёжность**: ошибки изолированы в отдельных подзадачах.
- **Отслеживание бюджета**: затраты оцениваются по каждой подзадаче и суммируются.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Создание агента планирования со структурированным выводом

Агент планирования выступает в роли **координатора на стойке регистрации**. Получая высокоуровневый запрос на поездку, он создает структурированный `TravelPlan` — разбивая запрос на подзадачи, устанавливая приоритеты и выявляя зависимости, чтобы консьерж или исполнительный уровень могли выполнить работу.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Выполнение плана с помощью специализированных инструментов

После того как агент на стойке регистрации составил структурированный план, его выполняет **агент-консьерж**.
Каждый специализированный инструмент обрабатывает одну категорию подзадач (рейсы, отели, активности). Консьерж
перебирает подзадачи плана в порядке зависимости и направляет каждую из них в соответствующий инструмент.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Резюме

В этом уроке вы изучили **шаблон проектирования планирования** для ИИ-агентов:

1. **Декомпозиция задачи** — агент планирования на ресепшн разбивает сложный запрос на путешествие на структурированные подзадачи с использованием моделей Pydantic, назначая каждую специалисту с приоритетами и зависимостями.
2. **Структурированный вывод** — передавая `response_format`, агент возвращает проверенный объект `TravelPlan` вместо свободного текста, что делает последующую обработку надежной.
3. **Выполнение плана** — консьерж-агент последовательно выполняет подзадачи с помощью специализированных инструментов (`book_flight`, `reserve_hotel`, `book_activity`), чтобы реализовать план и сообщить результаты.

Этот шаблон разделяет *что делать* (планирование) и *как делать* (выполнение), делая агентов более модульными, тестируемыми и простыми для расширения.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведен с помощью службы автоматического перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на то, что мы стремимся к точности, пожалуйста, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Исходный документ на оригинальном языке следует рассматривать как авторитетный источник. Для критически важной информации рекомендуется профессиональный перевод человеком. Мы не несем ответственности за любые недоразумения или ошибки толкования, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
